# 08b · Vision Transformers `[local]` (fine-tuning study: Kaggle)

**Goal:** turn attention into a full **Vision Transformer (ViT)**. You'll build one from scratch, load real pretrained weights into *your own* code, look at what it attends to, and compare ViTs with CNNs on accuracy, speed, data needs, transfer and robustness.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Image → patches → tokens; why ViTs need more data; DeiT and Swin; how ViTs and CNNs differ |
| 📐 Math | Token counts, parameter and FLOP counts, position embeddings at new resolutions |
| 🔨 From scratch | Patch embedding; ViT-S/16 with pretrained weights loaded into your code; a mini-ViT trained on CIFAR-10 |
| 🚀 Real models | ViT, DeiT, Swin vs. ResNet and ConvNeXt on Imagenette; transfer to Oxford Pets (linear probe, and fine-tuning on Kaggle) |
| 📊 Evaluate | Attention maps and attention rollout; learned position embeddings |
| 🧪 Experiment | Shuffled patches, hidden patches, a higher resolution |

### Two run modes
- **On your Mac:** everything except the fine-tuning study runs in ~20–25 minutes (close other apps first). The fine-tuning cell runs a tiny smoke test (its numbers mean nothing).
- **On Kaggle (GPU):** `FULL` mode adds the real fine-tuning comparison and trains the mini-ViT longer. Attach your `cvlib` dataset and turn on *Internet* and *GPU T4*.

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import os, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torchvision import datasets
from torchvision.datasets import ImageFolder
from cvlib.datasets import cifar10, imagenette, cache_images, IMAGENETTE_CLASSES, IMAGENETTE_TO_IMAGENET
from cvlib.loaders import GPULoader, random_crop_flip, random_resized_crop_flip, center_resize, CIFAR_MEAN, CIFAR_STD
from cvlib.models import SimpleCNN
from cvlib.transformer import ViT, PatchEmbed
from cvlib.lora import count_params
from cvlib import train as cvtrain

device = env.device
FULL = env.platform == "kaggle" or os.environ.get("CV_FULL") == "1"
print("MODE:", "FULL" if FULL else "LOCAL (fine-tuning study is a smoke test)")

def sync():
    if device == "cuda": torch.cuda.synchronize()
    elif device == "mps": torch.mps.synchronize()

def free():
    if device == "cuda": torch.cuda.empty_cache()
    elif device == "mps": torch.mps.empty_cache()

---
# 🧠 1 · Intuition

### An image as a sentence of patches
A transformer eats a sequence of tokens. A ViT makes one from an image:
1. **cut** the 224×224 image into 16×16 patches → a 14×14 grid = **196 patches**
2. **flatten** each patch (16·16·3 = 768 numbers) and map it with one shared linear layer to a **token** of width $D$
3. **prepend a [CLS] token**: a learned "summary" token with no patch of its own; it collects information from all patches through attention
4. **add position embeddings**: one learned vector per position, so the model knows where each patch came from
5. run **L transformer blocks** (attention + MLP, from 08a)
6. classify from the final [CLS] token

That's the whole model. No convolutions (except as a fast way to do steps 1–2), no pooling.

### Why ViTs need more data than CNNs
A CNN has two assumptions *built in* (its **inductive biases**):
- **locality:** a pixel mostly relates to its neighbours (3×3 kernels)
- **translation equivariance:** a cat is a cat wherever it is (the same kernel slides everywhere)

A ViT assumes almost nothing: any patch can attend to any other from layer 1, and it has to **learn** that neighbouring patches matter from data. With little data, the CNN's assumptions are a big head start. With a lot (ImageNet-21k: 14M images; JFT: 300M), the ViT catches up and then wins, because its assumptions don't limit it. You'll measure the small-data side of this on CIFAR-10.

### DeiT: a ViT trained on ImageNet-1k alone
**Data-efficient image Transformers** showed that the training recipe matters as much as the data: heavy augmentation (RandAugment, Mixup, CutMix), regularization (stochastic depth), long schedules, plus **distillation**: an extra token learns to copy the predictions of a strong CNN teacher. Result: ViT-level accuracy from 1.3M images.

### Swin: a ViT shaped like a CNN
Plain ViTs keep one coarse 14×14 token grid in every layer, and global attention gets too expensive on large images. **Swin Transformer**:
- starts with **small patches** (4×4) for fine detail
- uses **windowed attention** with shifted windows (08a) so cost grows linearly with image size
- **merges patches** between stages (56×56 → 28×28 → 14×14 → 7×7), giving a CNN-like pyramid of feature maps

That pyramid is exactly what detection and segmentation heads expect (feature pyramids, module 06), so Swin became a popular backbone for dense tasks.

### How ViTs and CNNs behave differently
| | CNN | ViT |
|---|---|---|
| Sees at layer 1 | a 3×3 neighbourhood | the whole image |
| Built-in assumptions | locality, translation | almost none |
| Data needed from scratch | moderate | a lot (or DeiT tricks) |
| Relies on | local textures more | global shape more |
| Shuffled or missing patches | struggles | more robust (tested below) |
| Cost vs. resolution | linear in pixels | quadratic in tokens (global attention) |
| Change input size | just works | interpolate position embeddings |

**Today:** pure CNNs (ConvNeXt), pure ViTs and hybrids all reach similar accuracy at similar compute. ViTs dominate when you need **pretrained foundation features** (CLIP, DINOv2, SigLIP: module 10) and as the image encoder of **VLMs** (module 13), because transformers scale with data and mix easily with text tokens.

---
# 📐 2 · The math

### Tokens
Image $H \times W$, patch $P$: $N = \frac{H}{P} \cdot \frac{W}{P}$ patch tokens, plus 1 [CLS].

| Model | Patch | Input | Tokens |
|---|---|---|---|
| ViT-B/16 | 16 | 224 | 196 + 1 |
| ViT-B/16 | 16 | 384 | 576 + 1 (≈ 3× tokens, ≈ 3.5× FLOPs) |
| ViT-B/32 | 32 | 224 | 49 + 1 (cheap, less detail) |

### Parameters per block
Width $D$, MLP hidden $4D$: attention has $4D^2$ (q, k, v, output projection), the MLP $8D^2$ → **≈ 12 D² per block**. ViT-S: $D = 384$, 12 blocks → 12 × 12 × 384² ≈ 21M, plus embeddings and head ≈ 22M.

### FLOPs per block (multiply-adds)
$$\underbrace{12\,N D^2}_{\text{linear layers}} + \underbrace{2\,N^2 D}_{\text{attention scores and mixing}}$$
At 224px ($N = 197$, $D = 384$) the linear layers dominate (~6×). The $N^2$ term takes over only at high resolution or with small patches.

### Position embeddings at a new resolution
Pretrained at 224 → a 14×14 grid of position vectors. At 384 the grid is 24×24: **reshape the embeddings to 14×14×D, resize them like an image (bicubic) to 24×24, flatten back.** Fine-tuning briefly at the new size recovers any lost accuracy.

### Pre-norm block
$$x \leftarrow x + \text{MHA}(\text{LN}(x)) \qquad x \leftarrow x + \text{MLP}(\text{LN}(x))$$
LayerNorm *before* each sub-layer keeps a clean residual path from input to output, which makes deep transformers train stably (like ResNet's skips, module 04).

---
# 🔨 3 · From scratch

### 3.1 · Patch embedding: a conv with kernel = stride = patch size

In [ ]:
x = torch.randn(2, 3, 224, 224)
pe = PatchEmbed(224, 16, 3, 384)
tokens_conv = pe(x)                                                    # (2, 196, 384)

# the same thing by hand: cut into patches, flatten each, apply one linear layer
patches = x.unfold(2, 16, 16).unfold(3, 16, 16)                        # (2, 3, 14, 14, 16, 16)
patches = patches.permute(0, 2, 3, 1, 4, 5).reshape(2, 196, 3 * 16 * 16)
tokens_manual = patches @ pe.proj.weight.reshape(384, -1).T + pe.proj.bias
print("tokens:", tuple(tokens_conv.shape), f"| max diff conv vs. manual: {(tokens_conv - tokens_manual).abs().max():.2e}")

### 3.2 · ViT-S/16 in your own code, with pretrained weights
`cvlib/transformer.py` builds a ViT from the 08a parts. Its layer names match timm's, so a pretrained checkpoint loads straight in. If **your** model gives the same outputs as timm's, every piece is right: patch embedding, [CLS], positions, attention, MLP, LayerNorm.

We use `vit_small_patch16_224.augreg_in21k_ft_in1k`: ViT-S/16 pretrained on ImageNet-21k, fine-tuned on ImageNet-1k.

In [ ]:
TIMM_VIT = "vit_small_patch16_224.augreg_in21k_ft_in1k"
ref = timm.create_model(TIMM_VIT, pretrained=True).eval()
mine = ViT(img_size=224, patch_size=16, num_classes=1000, dim=384, depth=12, num_heads=6).eval()
print(mine.load_state_dict(ref.state_dict(), strict=True))
print(f"parameters: mine {count_params(mine)[1] / 1e6:.2f}M | timm {count_params(ref)[1] / 1e6:.2f}M")

cfg = timm.data.resolve_data_config({}, model=ref)
print("timm data config:", {k: cfg[k] for k in ("input_size", "mean", "std", "crop_pct")})
VIT_MEAN, VIT_STD = cfg["mean"], cfg["std"]                     # this checkpoint uses 0.5/0.5, not ImageNet's mean/std

xb = torch.randn(4, 3, 224, 224)
with torch.no_grad():
    print(f"max |mine − timm| logits: {(mine(xb) - ref(xb)).abs().max():.2e}")
del ref

In [ ]:
root = imagenette(160)
val = ImageFolder(root / "val")
Xv, yv = cache_images("imagenette160_val", [p for p, _ in val.samples], 224, labels=[c for _, c in val.samples])
print("Imagenette val:", Xv.shape)

def to_input(x_uint8, mean, std):
    """uint8 (B, H, W, 3) numpy → normalized float tensor on the device."""
    x = torch.from_numpy(np.array(x_uint8)).to(device).permute(0, 3, 1, 2).float() / 255
    return (x - torch.tensor(mean, device=device).view(1, 3, 1, 1)) / torch.tensor(std, device=device).view(1, 3, 1, 1)

info = timm.data.ImageNetInfo()
imagenet_names = [info.index_to_description(i).split(",")[0] for i in range(1000)]
mine = mine.to(device)
sample = [0, 1000, 2000, 3500]
with torch.no_grad():
    probs = mine(to_input(Xv[sample], VIT_MEAN, VIT_STD)).softmax(-1).cpu()
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, i, p in zip(axes, sample, probs):
    top = p.topk(3)
    ax.imshow(Xv[i]); ax.axis("off")
    ax.set_title("\n".join(f"{imagenet_names[j][:22]} {v:.2f}" for v, j in zip(top.values, top.indices)), fontsize=8)
plt.suptitle("your ViT, pretrained weights: top-3 ImageNet classes"); plt.tight_layout(rect=(0, 0, 1, 0.93)); plt.show()

### 3.3 · A mini-ViT trained from scratch vs. a CNN
CIFAR-10 images are 32×32, so use **4×4 patches** (8×8 = 64 tokens + [CLS]). The mini-ViT (width 192, 6 blocks, 3 heads) has ~1.8M parameters, ~6× more than module 03's CNN. Both get the same augmentation, optimizer and epochs, at **two training-set sizes**.

**Predict first:** which wins at 5,000 images? Does the gap shrink with 4× more data?

In [ ]:
Xc, yc = cifar10("train"); Xct, yct = cifar10("test")
SIZES = [5000, 50000] if FULL else [5000, 20000]
EPOCHS = 30 if FULL else 8

def mini_vit():
    return ViT(img_size=32, patch_size=4, num_classes=10, dim=192, depth=6, num_heads=3, mlp_ratio=2, fast=True)

scratch = []
te = GPULoader(Xct, yct, batch_size=500, mean=CIFAR_MEAN, std=CIFAR_STD, device=device)
for n in SIZES:
    idx = np.random.default_rng(0).choice(len(yc), n, replace=False)
    for name, make in [("CNN", SimpleCNN), ("mini-ViT", mini_vit)]:
        torch.manual_seed(0)
        tr = GPULoader(Xc[idx], yc[idx], batch_size=128, shuffle=True, augment=random_crop_flip,
                       mean=CIFAR_MEAN, std=CIFAR_STD, device=device, drop_last=True)
        model = make()
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=EPOCHS * len(tr), pct_start=0.1)
        t0 = time.perf_counter()
        h = cvtrain.fit(model, tr, te, nn.CrossEntropyLoss(label_smoothing=0.1), opt, epochs=EPOCHS, device=device,
                        batch_scheduler=sched, verbose=False)
        scratch.append(dict(train_images=n, model=name, params_M=count_params(model)[1] / 1e6, test_acc=h["val_acc"][-1],
                            train_acc=h["train_acc"][-1], seconds=time.perf_counter() - t0, curve=h["val_acc"]))
        print(f"{n:>6} images | {name:8s} | test acc {h['val_acc'][-1]:.3f} (train {h['train_acc'][-1]:.3f}) | {time.perf_counter() - t0:.0f}s")
        del model, tr; free()

df_s = pd.DataFrame(scratch)
fig, axes = plt.subplots(1, len(SIZES), figsize=(11, 3.4))
for ax, n in zip(axes, SIZES):
    for _, r in df_s[df_s.train_images == n].iterrows():
        ax.plot(range(1, EPOCHS + 1), r.curve, "o-", label=r.model)
    ax.set_title(f"{n:,} training images"); ax.set_xlabel("epoch"); ax.set_ylabel("test acc"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

What we measured on the Mac (8 epochs):
- The **CNN wins by a wide margin**: ~64% vs. ~46% at 5,000 images, despite having ~6× fewer parameters. Its built-in locality is worth a lot of images.
- **4× more data doesn't close the gap at this scale:** both models gain about the same amount. The ViT's assumptions-free design only pays off with millions of images, or DeiT-style heavy augmentation and hundreds of epochs.
- Look at **train vs. test accuracy**: they're nearly equal for the ViT. It isn't overfitting; it's *learning slowly*, because it first has to discover that neighbouring patches are related, which the CNN gets for free.
- The mini-ViT is also 3–4× slower per epoch here. Attention over 65 tokens isn't the problem; the large linear layers are.

This is why, in practice, you almost never train a ViT from scratch: you start from a **pretrained** one.

---
# 🚀 4 · Real models

### 4.1 · ViT, DeiT and Swin vs. CNNs on Imagenette, with no training
Same protocol as module 04: ImageNet-pretrained models, keep the 10 Imagenette outputs, take the largest. Each model gets its own normalization from its timm config. Throughput is measured at batch 32 on your device.

In [ ]:
MODELS = {
    "ResNet-50": "resnet50.a1_in1k",
    "ConvNeXt-T": "convnext_tiny.fb_in1k",
    "ViT-S/16": TIMM_VIT,
    "DeiT-S/16": "deit_small_patch16_224.fb_in1k",
    "Swin-T": "swin_tiny_patch4_window7_224.ms_in1k",
}
subset = torch.tensor(IMAGENETTE_TO_IMAGENET, device=device)

@torch.no_grad()
def predict_subset(model, X, mean, std, transform=None, bs=64):
    preds = []
    for s in range(0, len(X), bs):
        x = to_input(X[s:s + bs], mean, std)
        if transform is not None:
            x = transform(x)
        preds.append(model(x)[:, subset].argmax(1).cpu())
    return torch.cat(preds).numpy()

@torch.no_grad()
def throughput(model, bs=32, reps=5):
    x = torch.randn(bs, 3, 224, 224, device=device)
    model(x); sync()
    t0 = time.perf_counter()
    for _ in range(reps): model(x)
    sync()
    return bs * reps / (time.perf_counter() - t0)

def load(tname):
    m = timm.create_model(tname, pretrained=True).eval().to(device)
    c = timm.data.resolve_data_config({}, model=m)
    return m, c["mean"], c["std"]

zoo = []
for nice, tname in MODELS.items():
    m, mean, std = load(tname)
    t0 = time.perf_counter()
    acc = (predict_subset(m, Xv, mean, std) == yv).mean()
    zoo.append(dict(model=nice, kind="CNN" if nice in ("ResNet-50", "ConvNeXt-T") else "transformer",
                    params_M=count_params(m)[1] / 1e6, imagenette_acc=acc, imgs_per_s=throughput(m)))
    print(f"{nice:11s} acc {acc:.4f} | {time.perf_counter() - t0:.0f}s")
    del m; free()
df_zoo = pd.DataFrame(zoo)
print(df_zoo.round(3).to_string(index=False))

Measured on the Mac: **all five score 99.3–99.7%**. These classes are too easy to separate the models; the robustness tests in 🧪 will. Two things to notice:
- **ViT-S and DeiT-S share an identical architecture** (and the same speed); only their training data and recipe differ. Keep that pair in mind for the experiments.
- **Speed:** ViT-S/DeiT-S ran about as fast as ResNet-50 (90–125 images/s across our runs; laptop timings are noisy), while ConvNeXt-T and Swin-T were consistently the slowest (~65–85). A plain ViT is just big matrix multiplications, which GPUs love; Swin's window shuffling and ConvNeXt's 7×7 depthwise convs are less GPU-friendly. Parameter counts don't predict speed (module 04's lesson again).

### 4.2 · Transfer: linear probe on Oxford Pets
Module 05's first question: how good are the frozen features for a *new* task? Extract pooled features once per backbone (no training), then fit a logistic regression on top. Pets (37 breeds) is close to ImageNet.

In [ ]:
pets_tr = datasets.OxfordIIITPet(env.data_dir, split="trainval", download=True)
pets_te = datasets.OxfordIIITPet(env.data_dir, split="test", download=True)
Xp, yp = cache_images("pets_trainval", pets_tr, 256)
Xpt, ypt = cache_images("pets_test", pets_te, 256)
crop = center_resize(224, 0.875)

@torch.no_grad()
def features(model, X, mean, std, bs=64):
    out = []
    for s in range(0, len(X), bs):
        x = crop(to_input(X[s:s + bs], mean, std))
        out.append(model.forward_head(model.forward_features(x), pre_logits=True).float().cpu())
    return torch.cat(out)

def linear_probe(Ftr, ytr, Fte, yte, epochs=150, lr=1e-2, wd=1e-4):
    ytr, yte = torch.as_tensor(ytr), torch.as_tensor(yte)
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6
    head = nn.Linear(Ftr.shape[1], int(ytr.max()) + 1)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    for _ in range(epochs):
        loss = F.cross_entropy(head((Ftr - mu) / sd), ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (head((Fte - mu) / sd).argmax(1) == yte).float().mean().item()

probe = []
for nice in ["ResNet-50", "ViT-S/16", "DeiT-S/16", "Swin-T"]:
    m, mean, std = load(MODELS[nice])
    t0 = time.perf_counter()
    Ftr, Fte = features(m, Xp, mean, std), features(m, Xpt, mean, std)
    acc = linear_probe(Ftr, yp, Fte, ypt)
    probe.append(dict(model=nice, feature_dim=Ftr.shape[1], pets_probe_acc=acc))
    print(f"{nice:10s} feature dim {Ftr.shape[1]:>4} | linear-probe test acc {acc:.4f} | {time.perf_counter() - t0:.0f}s")
    del m, Ftr, Fte; free()

Measured: Swin-T 93.3%, ViT-S 92.9%, DeiT-S 91.9%, ResNet-50 91.7%. ViT-S's 384-dim features beat ResNet-50's 2,048-dim ones, and ViT-S beats DeiT-S, its architectural twin: the 14M-image ImageNet-21k pretraining shows up in transfer even though both scored the same on Imagenette. That's why ViTs pretrained on huge datasets became the default for **foundation models** (module 10): features useful for new tasks *without* fine-tuning.

### 4.3 · Fine-tuning study: ViT-S vs. ResNet-50 (Kaggle)
Full fine-tuning on Pets at 10% and 100% of the training data, using module 05's recipe (AdamW, warmup + one-cycle schedule, label smoothing, lower learning rate for the backbone). ViTs usually want a smaller learning rate and more weight decay than CNNs.

On your Mac this is a 1-epoch smoke test on 200 images (meaningless numbers); on Kaggle it's the real comparison (~25 min on a T4).

In [ ]:
FT = dict(fractions=(0.1, 1.0) if FULL else (0.1,), epochs={0.1: 15, 1.0: 6} if FULL else {0.1: 1},
          cap=None if FULL else 200, test_cap=None if FULL else 200, batch=48 if FULL else 16)
FT_LR = {"ResNet-50": 1e-3, "ViT-S/16": 3e-4}

def finetune(nice, frac):
    rng = np.random.default_rng(0)
    idx = np.concatenate([rng.permutation(np.nonzero(yp == c)[0])[:max(1, round(frac * (yp == c).sum()))] for c in range(37)])
    if FT["cap"]: idx = rng.choice(idx, FT["cap"], replace=False)
    te_idx = np.arange(len(ypt)) if FT["test_cap"] is None else rng.choice(len(ypt), FT["test_cap"], replace=False)
    m = timm.create_model(MODELS[nice], pretrained=True, num_classes=37)
    c = timm.data.resolve_data_config({}, model=m)
    tr = GPULoader(np.asarray(Xp[np.sort(idx)]), yp[np.sort(idx)], batch_size=FT["batch"], shuffle=True,
                   augment=random_resized_crop_flip(224), mean=c["mean"], std=c["std"], device=device, drop_last=True)
    te = GPULoader(np.asarray(Xpt[np.sort(te_idx)]), ypt[np.sort(te_idx)], batch_size=64, augment=crop, mean=c["mean"], std=c["std"], device=device)
    head_names = ("head.", "fc.")
    lr = FT_LR[nice]
    groups = [{"params": [p for n, p in m.named_parameters() if not n.startswith(head_names)], "lr": lr / 10},
              {"params": [p for n, p in m.named_parameters() if n.startswith(head_names)], "lr": lr}]
    opt = torch.optim.AdamW(groups, weight_decay=0.05)
    epochs = FT["epochs"][frac]
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g["lr"] for g in groups], total_steps=epochs * len(tr), pct_start=0.1)
    t0 = time.perf_counter()
    h = cvtrain.fit(m, tr, te, nn.CrossEntropyLoss(label_smoothing=0.1), opt, epochs=epochs, device=device,
                    batch_scheduler=sched, amp=device == "cuda", verbose=False)
    out = dict(model=nice, fraction=frac, n_train=len(idx), test_acc=h["val_acc"][-1], seconds=time.perf_counter() - t0)
    del m, tr, te; free()
    return out

ft = []
for frac in FT["fractions"]:
    for nice in ("ResNet-50", "ViT-S/16"):
        ft.append(finetune(nice, frac))
        print(ft[-1])
df_ft = pd.DataFrame(ft)
print(f"\n{'FULL' if FULL else 'SMOKE (meaningless numbers)'} fine-tuning results:")
print(df_ft.round(4).to_string(index=False))

On Kaggle, compare with the linear probe above: how much does fine-tuning add for each backbone, at 10% and at 100% of the data? Also note the training time per epoch: ViT-S and ResNet-50 have similar compute, but mixed precision (`amp`) helps the transformer more on NVIDIA GPUs because almost all its work is large matrix multiplications.

---
# 📊 5 · Evaluate: what does the ViT look at?

### 5.1 · Learned position embeddings
Nobody told the model that patches form a 2-D grid. Compare each position embedding with all the others (cosine similarity): if the model learned the geometry, each patch's map should light up around its own location, and along its row and column.

In [ ]:
pos = mine.pos_embed[0, 1:].detach().cpu()                         # drop [CLS]: (196, 384)
pos = F.normalize(pos, dim=-1)
sim = (pos @ pos.T).view(14, 14, 14, 14)
fig, axes = plt.subplots(4, 4, figsize=(6.5, 6.5))
for ax, (r, c) in zip(axes.flat, [(r, c) for r in (0, 4, 9, 13) for c in (0, 4, 9, 13)]):
    ax.imshow(sim[r, c], cmap="viridis"); ax.plot(c, r, "r.", ms=5); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("similarity of patch (row, col)'s position embedding to all 14×14 positions", fontsize=9); plt.tight_layout(); plt.show()

### 5.2 · Attention maps
Two views:
- **Last-layer [CLS] attention:** which patches the summary token reads from in the final block, per head.
- **Attention rollout:** attention only says how *one* layer mixes tokens. Rollout multiplies the attention matrices of all layers (adding the identity for the residual path) to estimate how much each input patch flows into the final [CLS] token.

In [ ]:
def rollout(attns):
    """attns: list of (B, heads, T, T). Average heads, add the residual, renormalize, multiply through the layers."""
    result = None
    for a in attns:
        a = a.mean(1)
        a = a + torch.eye(a.shape[-1], device=a.device)
        a = a / a.sum(-1, keepdim=True)
        result = a if result is None else a @ result
    return result[:, 0, 1:]                                         # [CLS] row, patch columns

mine.keep_attention(True)
ids = [10, 400, 1300, 1800, 2700, 3700]
with torch.no_grad():
    mine(to_input(Xv[ids], VIT_MEAN, VIT_STD))
attns = mine.attention_maps()
last_cls = attns[-1][:, :, 0, 1:]                                   # (B, heads, 196)
roll = rollout(attns)
mine.keep_attention(False)

def overlay(ax, img, heat, title):
    heat = heat.view(14, 14).float().cpu().numpy()
    heat = cv2.resize((heat - heat.min()) / (heat.max() - heat.min() + 1e-9), (224, 224), interpolation=cv2.INTER_CUBIC)
    ax.imshow(img); ax.imshow(heat, cmap="jet", alpha=0.45); ax.set_title(title, fontsize=8); ax.axis("off")

fig, axes = plt.subplots(len(ids), 5, figsize=(11, 2.3 * len(ids)))
for row, i in enumerate(ids):
    axes[row, 0].imshow(Xv[i]); axes[row, 0].axis("off"); axes[row, 0].set_title(IMAGENETTE_CLASSES[yv[i]], fontsize=8)
    overlay(axes[row, 1], Xv[i], last_cls[row].mean(0), "last layer, mean of heads")
    overlay(axes[row, 2], Xv[i], last_cls[row, 0], "last layer, head 0")
    overlay(axes[row, 3], Xv[i], last_cls[row, 5], "last layer, head 5")
    overlay(axes[row, 4], Xv[i], roll[row], "rollout (all layers)")
plt.tight_layout(); plt.show()

What this model shows:
- The **last-layer maps are dominated by a few bright background patches**: a sky corner, a spot on the floor, the same spots in every head. These are known **artifact tokens**: ViTs learn to use some unimportant background patches as scratch space for global information, and the [CLS] token reads from them. (The fix in newer models such as DINOv2 with registers: add a few extra "register" tokens to hold that information.)
- **Rollout** spreads over the object much better (the chainsaw, the dog, the fish), because it also counts the earlier layers, where patches gather information from the object.
- Self-supervised ViTs (DINO, module 10) give much cleaner maps that outline objects.

**Caution:** attention maps show where information *flows*, not *why* the model decided something. They're a sanity check, not an explanation. Grad-CAM (module 04) and occlusion tests (next) are more direct.

---
# 🧪 6 · Experiments
All tests use the pretrained models from 4.1 on 1,000 Imagenette validation images, with no training.

### Experiment A · Shuffle the patches
Cut each image into a grid of tiles and shuffle them randomly. Object parts survive inside each tile, but the global layout is destroyed.

**Predict:** at 4×4 tiles (56 px each) and 14×14 tiles (16 px each), which models hold up better: CNNs or transformers?

In [ ]:
EVAL_IDX = np.sort(np.random.default_rng(0).choice(len(yv), 1000, replace=False))
Xe, ye = Xv[EVAL_IDX], yv[EVAL_IDX]

def shuffle_tiles(grid, seed=0):
    g = torch.Generator().manual_seed(seed)
    def t(x):
        B, C, H, W = x.shape; s = H // grid
        tiles = x.unfold(2, s, s).unfold(3, s, s).reshape(B, C, grid * grid, s, s)
        tiles = tiles[:, :, torch.randperm(grid * grid, generator=g).to(x.device)]
        return tiles.view(B, C, grid, grid, s, s).permute(0, 1, 2, 4, 3, 5).reshape(B, C, H, W)
    return t

def hide_patches(frac, size=16, seed=0):
    g = torch.Generator().manual_seed(seed)
    def t(x):
        B, C, H, W = x.shape; n = H // size
        keep = (torch.rand(B, 1, n, n, generator=g) >= frac).float().to(x.device)
        return x * F.interpolate(keep, scale_factor=size, mode="nearest")   # hidden patches → 0 (the mean color after normalization)
    return t

tests = {"clean": None, "shuffle 2×2": shuffle_tiles(2), "shuffle 4×4": shuffle_tiles(4), "shuffle 14×14": shuffle_tiles(14),
         "hide 25%": hide_patches(0.25), "hide 50%": hide_patches(0.5), "hide 75%": hide_patches(0.75)}

demo = torch.as_tensor(np.asarray(Xe[:1])).permute(0, 3, 1, 2).float() / 255
fig, axes = plt.subplots(1, len(tests), figsize=(15, 2.4))
for ax, (name, t) in zip(axes, tests.items()):
    img = demo if t is None else t(demo)
    ax.imshow(img[0].permute(1, 2, 0).clamp(0, 1)); ax.set_title(name, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

robust = []
for nice, tname in MODELS.items():
    m, mean, std = load(tname)
    for test, t in tests.items():
        robust.append(dict(model=nice, test=test, acc=(predict_subset(m, Xe, mean, std, t) == ye).mean()))
    print(nice, "done")
    del m; free()
df_rob = pd.DataFrame(robust).pivot(index="model", columns="test", values="acc")[list(tests)]
print(df_rob.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, cols, title in [(axes[0], ["clean", "shuffle 2×2", "shuffle 4×4", "shuffle 14×14"], "shuffled tiles"),
                        (axes[1], ["clean", "hide 25%", "hide 50%", "hide 75%"], "hidden 16×16 patches")]:
    for nice in MODELS:
        ax.plot(cols, df_rob.loc[nice, cols], "o-" if nice in ("ResNet-50", "ConvNeXt-T") else "s--", label=nice)
    ax.set_title(title); ax.set_ylabel("accuracy"); ax.grid(alpha=0.3); ax.axhline(0.1, color="gray", lw=0.8)
axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()

How to read it (solid lines = CNNs, dashed = transformers; gray line = chance). What we measured:
- **ResNet-50 is the least robust on both tests**: ~84% with 75% of patches hidden (others 86–94%) and ~50% on 14×14 shuffled tiles (others 60–81%).
- **But "transformer vs. CNN" isn't the real dividing line.** ConvNeXt-T, a pure CNN, is about as robust as DeiT-S to hidden patches (~93%). And ViT-S, a transformer, drops almost as much as ResNet-50 at 75% hidden (~87%).
- **The training recipe explains more than the architecture.** DeiT, Swin and ConvNeXt were all trained with heavy augmentation, including **CutMix and random erasing**, which literally paste over or blank out parts of images. ViT-S (augreg) and ResNet-50 (a1) used different recipes. Same architecture (ViT-S vs. DeiT-S), different recipe → different robustness.
- **Swin** holds up best on shuffled tiles (~81%): with its small 4×4 patches and local windows, it leans more on local texture, which survives the shuffle.

Commonly repeated claims ("ViTs are more robust to occlusion") are partly about the *recipes* ViTs were published with. When you compare architectures, compare training recipes too. (Masked image modeling, module 10, makes robustness to missing patches an explicit training goal: hide 75% of a ViT's patches and train it to fill them in.)

### Experiment B · A higher input resolution
Interpolate ViT-S's 14×14 position embeddings to 20×20 and run at 320 px with **no retraining**. More tokens = more detail, but the model was never trained at this scale.

**Predict:** does accuracy go up, down, or stay the same? How much slower is it?

In [ ]:
def resize_pos_embed(model, new_grid):
    pe = model.pos_embed.data
    cls, grid = pe[:, :1], pe[:, 1:]
    old = int(grid.shape[1] ** 0.5)
    grid = grid.reshape(1, old, old, -1).permute(0, 3, 1, 2)
    grid = F.interpolate(grid, size=(new_grid, new_grid), mode="bicubic", align_corners=False)
    model.pos_embed = nn.Parameter(torch.cat([cls, grid.permute(0, 2, 3, 1).reshape(1, new_grid * new_grid, -1)], 1))
    model.patch_embed.grid, model.patch_embed.num_patches = new_grid, new_grid * new_grid
    return model

hi_root = imagenette(320)
hi_val = ImageFolder(hi_root / "val")
Xv320, _ = cache_images("imagenette320_val", [p for p, _ in hi_val.samples], 320, labels=[c for _, c in hi_val.samples])
Xv224b, _ = cache_images("imagenette320_val", [p for p, _ in hi_val.samples], 224, labels=[c for _, c in hi_val.samples])

vit = ViT(img_size=224, patch_size=16, num_classes=1000, dim=384, depth=12, num_heads=6, fast=True)
vit.load_state_dict(mine.state_dict()); vit = vit.eval().to(device)
res = []
for size, X in [(224, Xv224b), (320, Xv320)]:
    if size != 224:
        resize_pos_embed(vit, size // 16)
    t0 = time.perf_counter()
    acc = (predict_subset(vit, X[EVAL_IDX], VIT_MEAN, VIT_STD) == ye).mean()
    res.append(dict(input=size, tokens=(size // 16) ** 2 + 1, acc=acc, seconds=time.perf_counter() - t0))
print(pd.DataFrame(res).round(4).to_string(index=False))
del vit, mine; free()

Both rows use the same source photos (the 320 px Imagenette release), so only the input size changes. Measured: **the same accuracy** at both sizes (within a few images out of 1,000), and 320 px took ~2.3× longer (2× the tokens, plus the attention cost growing faster than linearly). Interpolating the position embeddings works: the model runs fine at a size it never saw. But these objects are large and easy, so the extra detail buys nothing here. When you need high resolution (documents, small objects), **fine-tune briefly at that resolution** after interpolating; that's how the `_384` checkpoints in timm are made.

---
# 📝 7 · Recap

- **ViT** = patch embedding (a conv with kernel = stride = patch) + [CLS] + position embeddings + transformer blocks + a head on [CLS]. You built it and loaded real weights into it.
- ~**12·D² parameters per block**; FLOPs ≈ 12·N·D² + 2·N²·D per block, so the linear layers dominate at 224 px.
- **Inductive bias:** CNNs assume locality and translation; ViTs learn them from data. From scratch with little data, CNNs win. Pretrained, ViTs match or beat them and transfer very well.
- **DeiT:** a strong recipe plus distillation makes ViTs work with ImageNet-1k only. **Swin:** windowed attention + a CNN-like pyramid, suited to detection and segmentation.
- ViTs handle **missing patches** gracefully (the idea behind masked image modeling), but cost grows quadratically with tokens. For a new resolution, **interpolate the position embeddings** and fine-tune.
- **Attention maps and rollout** show information flow, not explanations. Watch for artifact tokens.

### Interview questions
1. Walk through a ViT's forward pass for a 224×224 image with 16×16 patches. How many tokens, and what's the [CLS] token for?
2. Why do ViTs need more data than CNNs? How did DeiT get around it?
3. How does Swin Transformer differ from ViT, and why does that matter for detection and segmentation?
4. How do you run a ViT at a higher resolution than it was trained at?
5. Roughly how many parameters does a transformer block with width D have? Where do most of the FLOPs go at 224 px vs. 1024 px?
6. What is attention rollout, and why shouldn't you treat attention maps as explanations?
7. Why are ViTs more robust than CNNs to occluded patches?
8. CLS-token pooling vs. global average pooling of patch tokens: what's the difference?

**Next:** module 09 uses attention for detection: DETR's object queries cross-attend to image features, and Hungarian matching replaces NMS.